In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv; nproc

In [ ]:
!ls /usr/lib/x86_64-linux-gnu/ | grep -i -E 'nvidia|libEGL' | head -30; ls /usr/share/glvnd/egl_vendor.d/ 2>&1; ls /usr/lib/x86_64-linux-gnu/libnvidia* 2>/dev/null | head

In [ ]:
!pip install -q uv
!UV_CACHE_DIR=/tmp/uvcache uv venv -q -p 3.12 /tmp/venv
!UV_CACHE_DIR=/tmp/uvcache uv pip install -q --python /tmp/venv/bin/python "mujoco==3.3.7" scipy pyopengl pillow

In [ ]:
open('/tmp/misura.py', 'w').write('\nimport os, sys, time, glob\nsys.path.insert(0, glob.glob(\'/kaggle/input/**/robotica/src\', recursive=True)[0])\nimport mujoco, numpy as np\nfrom robotica.sim.so101_caramelle import costruisci\nm, d, ep = costruisci(np.random.default_rng(3))\nmujoco.mj_forward(m, d)\nr = mujoco.Renderer(m, 480, 640)\nfrom OpenGL import GL\nprint("GL_RENDERER:", GL.glGetString(GL.GL_RENDERER), "| GL_VENDOR:", GL.glGetString(GL.GL_VENDOR))\nfor nome, ombre in (("ombre accese", True), ("ombre spente", False)):\n    r.update_scene(d, camera="wrist_cam")\n    r.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = ombre\n    r.render()\n    t = time.perf_counter()\n    for _ in range(30):\n        r.update_scene(d, camera="wrist_cam")\n        r.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = ombre\n        img = r.render()\n    print(f"{nome}: {1000 * (time.perf_counter() - t) / 30:.1f} ms a fotogramma, media pixel {img.mean():.1f}")\n    from PIL import Image\n    Image.fromarray(img).save(f"/kaggle/working/{os.environ.get(\'ETICHETTA\', \'prova\')}_{\'ombre\' if ombre else \'senza\'}.png")\nr.close()\n')
!ETICHETTA=base MUJOCO_GL=egl PYOPENGL_PLATFORM=egl /tmp/venv/bin/python /tmp/misura.py 2>&1 | tail -15

In [ ]:
# Piano A: le librerie GL di NVIDIA da apt. Valgono solo se la versione coincide col driver.
import subprocess
ver = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0]
major = ver.split('.')[0]
print('driver', ver)
subprocess.run('apt-get -qq update > /dev/null 2>&1', shell=True)
for pacchetto in (f'libnvidia-gl-{major}', f'libnvidia-gl-{major}-server'):
    print(subprocess.run(f'apt-cache policy {pacchetto} | head -3', shell=True, capture_output=True, text=True).stdout)
    r = subprocess.run(f'apt-get -qq install -y --no-install-recommends {pacchetto}', shell=True, capture_output=True, text=True)
    print(pacchetto, 'esito', r.returncode, r.stderr[-300:])
    if r.returncode == 0:
        break
!ls /usr/share/glvnd/egl_vendor.d/; ls /usr/lib/x86_64-linux-gnu/ | grep -i -E 'libEGL_nvidia|libnvidia-eglcore'

In [ ]:
!ETICHETTA=apt MUJOCO_GL=egl PYOPENGL_PLATFORM=egl __EGL_VENDOR_LIBRARY_FILENAMES=/usr/share/glvnd/egl_vendor.d/10_nvidia.json /tmp/venv/bin/python /tmp/misura.py 2>&1 | tail -15

In [ ]:
# Piano B: dall'installer NVIDIA della versione ESATTA del driver, estratto e non installato.
# Solo le librerie GL/EGL, in una cartella privata: il sistema non si tocca.
!cd /tmp && (wget -q https://us.download.nvidia.com/tesla/{ver}/NVIDIA-Linux-x86_64-{ver}.run -O nv.run || wget -q https://us.download.nvidia.com/XFree86/Linux-x86_64/{ver}/NVIDIA-Linux-x86_64-{ver}.run -O nv.run); ls -la /tmp/nv.run
!cd /tmp && sh nv.run --extract-only --target /tmp/nvx > /dev/null && mkdir -p /tmp/nvlib && cp /tmp/nvx/libEGL_nvidia.so.{ver} /tmp/nvx/libnvidia-eglcore.so.{ver} /tmp/nvx/libnvidia-glsi.so.{ver} /tmp/nvlib/ && (cp /tmp/nvx/libnvidia-gpucomp.so.{ver} /tmp/nvlib/ 2>/dev/null; true) && ln -sf libEGL_nvidia.so.{ver} /tmp/nvlib/libEGL_nvidia.so.0 && ls /tmp/nvlib
open('/tmp/nvlib/10_nvidia.json', 'w').write('{"file_format_version": "1.0.0", "ICD": {"library_path": "/tmp/nvlib/libEGL_nvidia.so.0"}}')

In [ ]:
!ETICHETTA=run LD_LIBRARY_PATH=/tmp/nvlib:$$LD_LIBRARY_PATH MUJOCO_GL=egl PYOPENGL_PLATFORM=egl __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvlib/10_nvidia.json /tmp/venv/bin/python /tmp/misura.py 2>&1 | tail -15